[README](README.md) | [Introduction](Introduction.md) | [Datasets](Datasets.md) | Notebook

# How the Internet assigns and uses Autonomous Systems (ASes)

In [ ]:
name = "Landon Faber"
date = "03/12/2004"

In [1]:
# Setup: shared utilities and tier definitions
import bz2
import urllib.request
import gzip
import io
import json
import zipfile
from contextlib import contextmanager
from pathlib import Path
from IPython.display import Markdown, display


@contextmanager
def open_safe(filename, encoding="utf-8"):
    """Open a file for reading, transparently handling .gz, .bz2, and .zip compression."""
    path = Path(filename)
    suffix = path.suffix.lower()
    if suffix == ".gz":
        with gzip.open(path, "rt", encoding=encoding) as f:
            yield f
    elif suffix == ".bz2":
        with bz2.open(path, "rt", encoding=encoding) as f:
            yield f
    elif suffix == ".zip":
        with zipfile.ZipFile(path) as zf:
            with zf.open(zf.namelist()[0]) as raw:
                yield io.TextIOWrapper(raw, encoding=encoding)
    else:
        with path.open(encoding=encoding) as f:
            yield f


TIERS = [
    ("edge",           1,     1),
    ("transit small",  2,     10),
    ("transit middle", 11,    1000),
    ("transit large",  1001,  10000),
    ("transit huge",   10001, -1),
]

---

## Task 2: ASN Classified by Customer Cone Size

Fill in the missing code in the cells below to produce **Table 1**.

### Data format

In `data/20260501.ppdc-ases.txt.bz2`, lines starting with `#` are comments.
All other lines start with an ASN followed by the ASNs in its customer cone.
The cone size for that ASN is the number of space-separated tokens on the line minus 1.

```
# This is a comment
# 23's customer cone size is 3 and includes (23, 4, 1)
23 23 4 1
# 1's customer cone size is 1 — it only includes itself
1 1
```

### Table 1 format

|           tier | range        | number of ASNs |   percentage |
| -------------: | ------------ | -------------: | -----------: |
|           edge | 1            |        [total] | [percentage] |
|  transit small | 2..10        |        [total] | [percentage] |
| transit middle | 11..1000     |        [total] | [percentage] |
|  transit large | 1001..10000  |        [total] | [percentage] |
|   transit huge | 10001..[max] |        [total] | [percentage] |

Replace `[max]` with the actual maximum cone size found in the data.
Percentages are formatted to one decimal place.

In [29]:
AS_CONE_URL = "http://rook-ceph-rgw-nautiluss3.rook/caida/as-relationships/20260501.ppdc-ases.txt.bz2"
AS_CONE_PATH = Path("data/20260501.ppdc-ases.txt.bz2")

def classify(size: int) -> str:

    for label, lo, hi in TIERS:
        if hi == -1:
            if size >= lo:
                return label
        elif lo <= size <= hi:
            return label
    return "unknown"

print(f"Downloading {AS_CONE_URL} ...")
AS_CONE_PATH.parent.mkdir(parents=True, exist_ok=True)  # ensure 'data/' exists
urllib.request.urlretrieve(AS_CONE_URL, AS_CONE_PATH)
if not AS_CONE_PATH.exists(): 
    print (f"Unable to save file {AS_CONE_PATH}")
    exit() 

# Build a dict mapping each ASN to its customer cone size
asn_to_size = {} 
with open_safe(AS_CONE_PATH) as fin:
    for line in fin:
        line = line.strip()
        if not line or line.startswith("#"):
            continue

        cone = line.split()
        asn = cone[0]
        asn_to_size[asn] = len(cone) - 1
        
print(f"Number of ASNs: {len(asn_to_size)}")


total = len(asn_to_size)
max_size = max(asn_to_size.values()) if asn_to_size else 0

tier_counts = {label: 0 for label, _, _ in TIERS}
for size in asn_to_size.values():
    tier_counts[classify(size)] += 1

header = "| tier | range | number of ASNs | percentage |"
sep    = "| ---: | ----- | ---: | ---: |"
table_1_rows = [header, sep]

for label, lo, hi in TIERS:
    # YOUR CODE HERE
    # Compute the range string, count, and percentage 
    # for this tier, and append a row
    if lo == hi:
        range_str = f"{lo}"
    elif hi == -1:
        range_str = f"{lo}..{max_size}"
    else:
        range_str = f"{lo}..{hi}"

    count = tier_counts[label]

    pct = count / total * 100
    
    table_1_rows.append(f"| {label} | {range_str} | {count} | {pct:.1f}% |")
table_1 = "\n".join(table_1_rows)

Number of ASNs: 79644


In [30]:
display(Markdown(table_1))
Path("tables").mkdir(exist_ok=True)
Path("tables/asn-cone-tiers.md").write_text(table_1, encoding="utf-8")

| tier | range | number of ASNs | percentage |
| ---: | ----- | ---: | ---: |
| edge | 1 | 67090 | 84.2% |
| transit small | 2..10 | 9977 | 12.5% |
| transit middle | 11..1000 | 2511 | 3.2% |
| transit large | 1001..10000 | 55 | 0.1% |
| transit huge | 10001..56415 | 11 | 0.0% |

279

### Question 1

What percentage of ASNs are edge ASes (customer cone size of 1)? What does this suggest about the structure of the Internet?

84.2% of the ASNs are edge ASes. This suggests that the vast majoirty of the internet structure are ASes that are endpoint ASNs.

### Question 2

How large is the maximum customer cone? What does this tell you about the most influential ASes on the Internet?

The maximum customer cone is 56415.This tells us that the most influential ASes on the internet route a ton of traffic through as a singular AS. In this case, the largest AS in this dataset can route to 71% of the total ASes.

### Question 3

How do the proportions of edge, small transit, and large transit ASes compare? What does this distribution reveal about how ASes are organized hierarchically?

The proportion of edge, small, and large transit ASes compares in a way that makes general sense. There are more ASes the lower you go in range. It makes sense that the largest number of ASNs would be at the edge, as they are at the end of the customer chain. 

---

## Task 3: How Are ASN Tiers Divided Across Countries?

Fill in the missing code in the cells below to produce **Table 2**.

- Use the tier classification from Task 2 (`classify()` and `TIERS`).
- Use `data/as2org.jsonl` to map each ASN to its organization's 2-letter country code.
  Each line is a JSON object with a `"country"` field and a `"members"` list of ASN strings.
- Find the top 4 countries by total ASN count across all tiers.
  All other countries map to **other**.
- The top-4 country codes become the dynamic column headers (e.g. `US`, `CN`).

### Table 2 format

| name           | 1st           | 2nd           | 3rd           | 4th           | other         |
| -------------- | ------------- | ------------- | ------------- | ------------- | ------------- |
| edge           | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) |
| transit small  | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) |
| transit middle | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) |
| transit large  | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) |
| transit huge   | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) |

Replace `1st`/`2nd`/`3rd`/`4th` with the actual 2-letter country codes.
Each cell shows `total (X.X%)`, where `X.X%` is that country's share of the tier's total.

In [38]:
AS2ORG_URL = "http://rook-ceph-rgw-nautiluss3.rook/caida/as2org/as2org.jsonl"
AS2ORG_PATH = Path("data/as2org.jsonl")

print(f"Downloading {AS2ORG_URL} ...")
AS2ORG_PATH.parent.mkdir(parents=True, exist_ok=True)  # ensure 'data/' exists
urllib.request.urlretrieve(AS2ORG_URL, AS2ORG_PATH)
if not AS2ORG_PATH.exists(): 
    print (f"Unable to save file {AS2ORG_PATH}")
    exit() 
    
asn_to_country = {}
i = 0
with open_safe(AS2ORG_PATH) as fin:
    for line in fin:    
        line = line.strip()
        if not line:
            continue

        jline = json.loads(line)
        country = jline["country"]
        for asn in jline["members"]:
            asn_to_country[asn] = country

print (f"number of ASNs with countries: {len(asn_to_country)}")

number of ASNs with countries: 120793


In [102]:
from collections import defaultdict

# Count ASNs per (tier, country) pair
tier_country_counts = defaultdict(lambda: defaultdict(int))
country_totals = defaultdict(int)

for asn, size in asn_to_size.items():
    tier = classify(size)
    country = asn_to_country.get(asn, "unknown")
    tier_country_counts[tier][country] += 1
    country_totals[country] += 1

# Find top-4 countries by total ASN count
top4 = [c for c, _ in sorted(country_totals.items(), key=lambda x: -x[1])[:4]]
columns = top4 + ["other"]

# Build Markdown table
header = "| tier | " + " | ".join(columns) + " |"
sep    = "| --- | " + " | ".join(["---"] * len(columns)) + " |"
table_2_rows = [header, sep]

# YOUR CODE HERE: for each (label, _, _) in TIERS, append a formatted row to table_2_rows
#   each cell: f"{n} ({pct:.1f}%)" where pct is the country's share of that tier's total
#   sum counts for countries not in top4 into "other"

for label in TIERS:
    total = 0
    OT_tot = 0
    US_tot = tier_country_counts[label[0]]["US"]
    BR_tot = tier_country_counts[label[0]]["BR"]
    RU_tot = tier_country_counts[label[0]]["RU"]
    IN_tot = tier_country_counts[label[0]]["IN"]
    
    for country in tier_country_counts[label[0]]:
        if country != "US" and country != "BR" and country != "RU" and country != "IN":
            OT_tot += tier_country_counts[label[0]][country]
        total += tier_country_counts[label[0]][country]
        
    table_2_rows.append(f"| {label[0]} | {US_tot} , {(US_tot/total*100):.1f}% | {BR_tot} , {(BR_tot/total*100):.1f}% | {RU_tot}  , {(RU_tot/total*100):.1f}%| {IN_tot} , {(IN_tot/total*100):.1f}% | {OT_tot} , {(OT_tot/total*100):.1f}%")

table_2 = "\n".join(table_2_rows)

In [103]:
display(Markdown(table_2))

| tier | US | BR | RU | IN | other |
| --- | --- | --- | --- | --- | --- |
| edge | 16587 , 24.7% | 5943 , 8.9% | 4140  , 6.2%| 2475 , 3.7% | 37945 , 56.6%
| transit small | 1582 , 15.9% | 1901 , 19.1% | 688  , 6.9%| 315 , 3.2% | 5491 , 55.0%
| transit middle | 414 , 16.5% | 350 , 13.9% | 127  , 5.1%| 66 , 2.6% | 1554 , 61.9%
| transit large | 10 , 18.2% | 7 , 12.7% | 8  , 14.5%| 3 , 5.5% | 27 , 49.1%
| transit huge | 7 , 63.6% | 0 , 0.0% | 0  , 0.0%| 0 , 0.0% | 4 , 36.4%

### Question 4

Which countries have the most transit huge ASNs? What does this tell you about where Internet infrastructure is concentrated?

The US has the most transit huge ASNs; this tells us that the Internet infrastructure is routed on a large scale through the US. This could also be because of the US's geographic size; other countries have more people, but are generally more concentrated in land area. The US is very spread out over the whole country and, as such, has more space to cover.

### Question 5

What proportion of all ASNs fall in the "other" category? What does this suggest about the geographic distribution of the global Internet?

Roughly 55 - 60% of all ASNs are in the "other" category. This suggests that over half of the internets ASNs do not route through the top four countries.

### Question 6

Do the same countries dominate across all AS tiers (edge, transit small, transit huge)? What patterns do you observe across the rows?

The US has the most ASes across all tiers compared to the other three countries, except for in-transit small. Overall, the US has the most; the next three countries have less than the previous country. This makes sense as they were the top four countries.